# Step 3: data plumbing
`PairDataset` (one pair as raw strings + label) and `make_collate(vocab)` (list of pairs → `(ids_a, len_a, ids_b, len_b, y)`) live in `11_data_plumbing.py`; this notebook tests them. Train loaders shuffle; val/test loaders must not (predictions have to line up with the DataFrame rows for `review_slices`).

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from importlib import import_module
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import torch
from torch import nn

from helpers.pairs import load_pairs

_vocab_step = import_module("siamese_model.01_vocab_step")       # digit-prefixed filenames -> can't use `from ... import`
build_vocab = _vocab_step.build_vocab
PAD_ID, UNK_ID = _vocab_step.PAD_ID, _vocab_step.UNK_ID
encode_batch = import_module("siamese_model.03_encode_batch_step").encode_batch
Encoder = import_module("siamese_model.08_encoder_step").Encoder

In [2]:
from torch.utils.data import DataLoader

# PairDataset / make_collate live in 11_data_plumbing.py (training imports them); this notebook tests them
_data_step = import_module("siamese_model.11_data_plumbing")
PairDataset, make_collate = _data_step.PairDataset, _data_step.make_collate

In [3]:
ot_train, ot_train_label = load_pairs("overture", "train")   # only the train split: the vocab is built from train
ds = PairDataset(ot_train, ot_train_label)
assert len(ds) == len(ot_train)
assert ds[0] == (ot_train.address_a.iloc[0], ot_train.address_b.iloc[0], ot_train_label.iloc[0])

vocab = build_vocab(ot_train)

In [4]:
train_loader = DataLoader(ds, batch_size=64, shuffle=False, collate_fn=make_collate(vocab))

ids_a, len_a, ids_b, len_b, y = next(iter(train_loader))   # peek at ONE batch
print(ids_a.shape, len_a.shape, ids_b.shape, y.shape, y.dtype)
assert ids_a.shape[0] == 64 and y.shape == (64,) and y.dtype == torch.float32
assert y.tolist() == ot_train_label.iloc[:64].astype(float).tolist()
# torch.Size([64, T_a]) torch.Size([64]) torch.Size([64, T_b]) torch.Size([64]) torch.float32

torch.Size([64, 19]) torch.Size([64]) torch.Size([64, 24]) torch.Size([64]) torch.float32


## Diagnostics

**Empty addresses after normalisation (2026-10-20, for the Step 3 encoder).** Question: does any pair have an empty / whitespace-only address on either side? (A length-0 string crashes `pack_padded_sequence`.) Result: **0** in every split of both datasets (Overture train 202,568 / val 41,865 / test 43,844; Stamford 15,197 / 977 / 764), and 0 NaN. A drop filter was added before the pair files are written (here §8b, and `Stamford/03_convert_to_shared_schema.py`) as a guard for future rebuilds; it drops nothing, so the pair files and every cache are unchanged (44 caches re-checked with `check_cache`). Query:
```python
normalize = import_module("siamese_model.01_vocab_step").normalize
for ds in ("overture", "stamford"):
    for split in ("train", "val", "test"):
        df, _ = load_pairs(ds, split)
        print(ds, split, (df.address_a.map(normalize) == "").sum() + (df.address_b.map(normalize) == "").sum())
```